# 🏛️ Architecture & Design Patterns — Hands-On Lab## Build the patterns that production GenAI systems are made of.**Setup:** `pip install openai numpy`**Cost:** ~$0.30 to run everything**Programs:** Semantic cache, model routing, fallback + MapReduce + refinement, circuit breaker + token budget, embedding pipeline, hybrid storage + freshness.

In [ ]:
# SETUP
from openai import OpenAI
import numpy as np, time, json, hashlib, random, re, math
from collections import defaultdict
from datetime import datetime, timedelta

client = OpenAI()

def ask(prompt, model='gpt-4o-mini', temperature=0):
    r = client.chat.completions.create(model=model, temperature=temperature,
        messages=[{'role':'user','content':prompt}])
    return r.choices[0].message.content.strip(), r.usage.total_tokens

def embed(texts):
    if isinstance(texts, str): texts = [texts]
    r = client.embeddings.create(model='text-embedding-3-small', input=texts)
    return [np.array(d.embedding) for d in r.data]

def cosine(a, b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

print('Setup complete!')

---# 1️⃣ Semantic Cache (Cache by Meaning, Not Exact Words)**Analogy:** A smart receptionist who knows 'Where is the bathroom?' and 'Can you direct me to the restroom?' are the same question — no need to ask the manager again.**What we build:** Cache that matches by MEANING (embedding similarity). 'Refund policy?' and 'How do returns work?' get the same cached answer.

In [ ]:
# SEMANTIC CACHE: Same meaning = same cached answer

class SemanticCache:
    def __init__(self, threshold=0.93):
        self.entries = []  # [{embedding, question, answer}]
        self.threshold = threshold
        self.hits = 0
        self.misses = 0
    
    def get(self, question):
        q_emb = embed(question)[0]
        for entry in self.entries:
            sim = cosine(q_emb, entry['embedding'])
            if sim >= self.threshold:
                self.hits += 1
                return entry['answer'], sim
        self.misses += 1
        return None, 0
    
    def set(self, question, answer):
        self.entries.append({
            'embedding': embed(question)[0],
            'question': question,
            'answer': answer,
        })

cache = SemanticCache(threshold=0.92)

queries = [
    'What is your refund policy?',
    'How do returns work?',           # Same meaning!
    'Can I get my money back?',       # Same meaning!
    'What is the Pro plan price?',    # Different topic
    'How much does Pro cost?',        # Same as above!
]

print('SEMANTIC CACHE')
print('=' * 60)

for q in queries:
    answer, sim = cache.get(q)
    if answer:
        print(f'  HIT  [{sim:.3f}] {q:40s} → cached')
    else:
        answer, _ = ask(q)  # Actually call LLM
        cache.set(q, answer)
        print(f'  MISS          {q:40s} → called LLM')

total = cache.hits + cache.misses
print(f'\n  Hit rate: {cache.hits}/{total} = {cache.hits/total*100:.0f}%')
print(f'  LLM calls saved: {cache.hits} (= {cache.hits} x $0.001 saved)')
print(f'  At 100K queries/day with 40% hit rate: save $40/day')

---# 2️⃣ MapReduce + Iterative Refinement**MapReduce:** Split 500-page book into chapters → summarize each → combine summaries. Like splitting a huge delivery across smaller trucks.**Refinement:** Write draft → critic reads it → rewrite → critic again. Like a writer-editor loop. Max 3 rounds.

In [ ]:
# MAPREDUCE: Split big tasks into parallel pieces + combine
# REFINEMENT: Generate → critique → improve → repeat

# ---------- MAPREDUCE ----------
print('MAPREDUCE (divide and conquer)')
print('=' * 60)

# Simulate a long document (10 'chapters')
chapters = [
    f'Chapter {i+1}: This chapter covers topic {i+1} with many important details about aspect {chr(65+i)} of the business.'
    for i in range(10)
]

# MAP: summarize each chapter (could run in parallel)
print(f'\n  MAP: Summarizing {len(chapters)} chapters...')
summaries = []
for i, chapter in enumerate(chapters[:5]):  # Doing 5 for demo cost
    summary, _ = ask(f'Summarize in 1 sentence: {chapter}')
    summaries.append(summary)
    print(f'    Chapter {i+1}: {summary[:50]}...')

# REDUCE: combine summaries
print(f'\n  REDUCE: Combining {len(summaries)} summaries...')
combined = '\n'.join(summaries)
final, _ = ask(f'Combine these summaries into one 2-sentence overview:\n{combined}')
print(f'    Final: {final[:80]}...')

print(f'\n  MAP used cheap model (extract). REDUCE used smart model (synthesize).')
print(f'  Run MAP in parallel = 10 chapters take same time as 1.')

# ---------- ITERATIVE REFINEMENT ----------
print(f'\n\nITERATIVE REFINEMENT (write → critique → improve)')
print('=' * 60)

task = 'Write a 3-sentence product description for a smart water bottle that tracks hydration.'

# Round 1: First draft
draft, _ = ask(task)
print(f'\n  Draft v1: {draft[:80]}...')

# Round 2: Critique + improve
critique, _ = ask(f'Critique this draft. List 2 specific improvements needed:\n{draft}')
print(f'  Critique: {critique[:80]}...')

improved, _ = ask(f'Improve this based on feedback:\nDraft: {draft}\nFeedback: {critique}')
print(f'  Draft v2: {improved[:80]}...')

# Round 3: Final check
check, _ = ask(f'Is this good enough? Any final tweaks?\n{improved}\nRate 1-10 and suggest one fix if needed.')
print(f'  Final check: {check[:80]}...')

print(f'\n  3 rounds is the sweet spot. After 3, diminishing returns.')

---# 3️⃣ Token Budget Manager + Circuit Breaker**Token Budget:** Like packing a suitcase — limited space. System prompt (500) + context (3000) + history (2000) + query (200) + answer space (2000) = 7700 total. Plan before you code.**Circuit Breaker:** Like a fuse. After 5 failures, stop trying (open). Wait 60s. Try one request (half-open). If works, resume.

In [ ]:
# TOKEN BUDGET: Plan your context window allocation
# CIRCUIT BREAKER: Handle provider outages

def plan_budget(max_context=8192):
    budget = {
        'System prompt': 500,
        'Retrieved docs': 3000,
        'Chat history': 2000,
        'User message': 200,
        'Response space': 2000,
    }
    total = sum(budget.values())
    remaining = max_context - total
    
    print(f'TOKEN BUDGET ({max_context:,} context window)')
    print('=' * 55)
    for name, tokens in budget.items():
        pct = tokens/max_context*100
        bar = '#' * int(pct/2)
        print(f'  {name:20s} {tokens:5,} tok ({pct:4.1f}%) {bar}')
    print(f'  {"REMAINING":20s} {remaining:5,} tok ({remaining/max_context*100:.1f}%)')
    if remaining < 0:
        print(f'  🔴 OVER BUDGET by {-remaining} tokens!')
    return budget

plan_budget(8192)
print()
plan_budget(128000)  # GPT-4o

# CIRCUIT BREAKER
print(f'\n\nCIRCUIT BREAKER')
print('=' * 55)

class CircuitBreaker:
    def __init__(self, threshold=3, timeout=10):
        self.failures = 0
        self.threshold = threshold
        self.state = 'CLOSED'
        self.last_fail = 0
        self.timeout = timeout
    
    def call(self, func, fallback):
        if self.state == 'OPEN':
            if time.time() - self.last_fail > self.timeout:
                self.state = 'HALF-OPEN'
                print(f'  🔄 HALF-OPEN: testing...')
            else:
                print(f'  ⚡ OPEN: immediate fallback')
                return fallback()
        try:
            result = func()
            if self.state == 'HALF-OPEN':
                self.state = 'CLOSED'; self.failures = 0
                print(f'  ✅ Recovered → CLOSED')
            return result
        except:
            self.failures += 1; self.last_fail = time.time()
            if self.failures >= self.threshold:
                self.state = 'OPEN'
                print(f'  🔴 OPEN after {self.failures} failures')
            else:
                print(f'  ⚡ Failure {self.failures}/{self.threshold}')
            return fallback()

cb = CircuitBreaker(threshold=3, timeout=5)
for i in range(5):
    result = cb.call(
        lambda: (_ for _ in ()).throw(Exception('down')),  # Always fails
        lambda: 'fallback answer'
    )
    print(f'  Call {i+1}: state={cb.state}, result={result[:20]}')

---# 4️⃣ Embedding Pipeline + Data Freshness**Analogy:** A bookstore: overnight truck for big shipments (batch), bike messenger for urgent deliveries (real-time). Content hashing prevents processing duplicates.**What we build:** Batch pipeline with deduplication + freshness checker.

In [ ]:
# EMBEDDING PIPELINE: Batch + dedup + freshness

class EmbeddingPipeline:
    def __init__(self):
        self.seen_hashes = set()
        self.stats = {'processed': 0, 'skipped': 0}
        self.chunks = []
    
    def chunk_text(self, text, size=200, overlap=30):
        words = text.split()
        chunks = []
        for i in range(0, len(words), size - overlap):
            chunks.append(' '.join(words[i:i+size]))
        return chunks
    
    def process_document(self, text, metadata):
        chunks = self.chunk_text(text)
        new_chunks = []
        for i, chunk in enumerate(chunks):
            h = hashlib.md5(chunk.encode()).hexdigest()
            if h in self.seen_hashes:
                self.stats['skipped'] += 1
                continue
            self.seen_hashes.add(h)
            new_chunks.append({
                'text': chunk, 'hash': h,
                'metadata': {**metadata, 'chunk_index': i},
            })
            self.stats['processed'] += 1
        self.chunks.extend(new_chunks)
        return len(new_chunks)

print('EMBEDDING PIPELINE')
print('=' * 60)

pipeline = EmbeddingPipeline()

docs = [
    ('Refund policy: 30 days for standard. ' * 20, {'source': 'policy.pdf'}),
    ('Pricing: Pro $49/mo Enterprise $199/mo. ' * 20, {'source': 'pricing.pdf'}),
    ('Refund policy: 30 days for standard. ' * 20, {'source': 'policy_v2.pdf'}),  # Duplicate!
]

for text, meta in docs:
    count = pipeline.process_document(text, meta)
    print(f'  {meta["source"]:20s}: {count} new chunks')

print(f'\n  Stats: {pipeline.stats}')
print(f'  Dedup saved {pipeline.stats["skipped"]} redundant embeddings')

# FRESHNESS CHECK
print(f'\n\nDATA FRESHNESS CHECK')
print('=' * 60)

doc_dates = [
    {'name': 'policy.pdf', 'updated': datetime(2024, 1, 15)},
    {'name': 'pricing.pdf', 'updated': datetime.now() - timedelta(days=5)},
    {'name': 'faq.pdf', 'updated': datetime(2023, 6, 1)},
]

cutoff = datetime.now() - timedelta(days=90)
for doc in doc_dates:
    stale = doc['updated'] < cutoff
    status = '🔴 STALE' if stale else '🟢 Fresh'
    print(f'  {status} {doc["name"]:15s} updated {doc["updated"].date()}')

stale_count = sum(1 for d in doc_dates if d['updated'] < cutoff)
print(f'\n  {stale_count}/{len(doc_dates)} documents need re-indexing')
print(f'\n  Strategy: incremental (only changed docs) for daily updates.')
print(f'  Full re-index when changing embedding model or chunking strategy.')

---# 🏁 Summary — Architecture Patterns Cheat Sheet| Pattern | When to Use | Cost Impact ||---------|-------------|-------------|| **Semantic Cache** | Repeated similar queries | Save 30-40% of LLM calls || **Model Routing** | Mix of simple + complex queries | Save 50-70% of cost || **Fallback Chain** | Production reliability | Near-zero downtime || **MapReduce** | Documents > context window | Process unlimited length || **Iterative Refinement** | Quality-critical outputs | 2-3x cost, much better quality || **Circuit Breaker** | Provider outages | Fast failure recovery || **Token Budget** | Every LLM app | Prevent context overflow || **Dedup Pipeline** | Any embedding pipeline | Save 10-30% storage + cost |**Rule:** Build these patterns INTO your architecture from day 1. Retrofitting is 5x harder.